# 07 — JEPA × ewm-state-machine: six demos in one cooperation story

This notebook consolidates the six [`ewm-jepa`](https://github.com/alexmy21/ewm-jepa) demos into a single cooperation narrative. One actor on each side:

```text
V-JEPA (encoder / predictor)          ewm-state-machine (the Rust lattice)
───────────────────────────           ────────────────────────────────────
continuous patch encodings z  ──Quantizer──▶  tid streams
                                                  │  ewm-scene ingest
                                                  │  G1/G2/G3 (three LUTs)
                                                  │  ewm-app [UM] chain
                                                  │  ewm-ops operational graph
                                                  │  ewm-scene materialize
predicted encodings ◀──decode── restored tids ◀────┘
```

| Section | Demo (source notebook) | What it proves |
| - | - | - |
| §1 | Pipeline validation (01) | IICA: idempotent, immutable, content-addressed |
| §2 | Three-LUT unification (02) | G1/G2/G3 bootstrap views disambiguate |
| §3 | Recursive IICA chain (03) | chained [UM]s compose; fixed point in one pass |
| §4 | [UM]-Net agent network (04) | fan-out by reference, deterministic firing |
| §5 | Holographic memory (05) | temporal pyramid D/R/N balance |
| §6 | Grounding proof (06) | fidelity round-trip + one-sided gating |

In [1]:
# ── 0. One environment, two projects ─────────────────────────────────────
import json
import os
import pathlib
import subprocess
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent


def find_project(marker, start, max_up=5):
    """Find a project root by a relative marker path, searching the start
    directory, its ancestors, and their immediate subdirectories — so the
    two repos can live as siblings or anywhere nearby."""
    parts = pathlib.Path(marker).parts
    for anc in [start, *list(start.parents)[:max_up]]:
        roots = [anc]
        if anc.is_dir():
            roots += [p for p in anc.iterdir() if p.is_dir()]
        for root in roots:
            for sub in ((), ('ewm-jepa',), ('ewm_jepa',), ('ewm-state-machine',)):
                p = root.joinpath(*sub, *parts)
                if p.exists():
                    return root.joinpath(*sub).resolve()
    return None


EWM_SM = find_project('crates/ewm-app/Cargo.toml', ROOT)
if EWM_SM is None:
    EWM_SM = pathlib.Path(os.environ.get('EWM_SM', str(ROOT)))
if not (EWM_SM / 'crates' / 'ewm-app').exists():
    raise RuntimeError(
        'ewm-state-machine not found. Clone it nearby or set EWM_SM=/path/to/ewm-state-machine.'
    )

EWM_JEPA = find_project('ewm_jepa/__init__.py', ROOT)
if EWM_JEPA is None:
    EWM_JEPA = pathlib.Path(os.environ.get('EWM_JEPA', str(ROOT.parent / 'ewm-jepa')))
if not (EWM_JEPA / 'ewm_jepa' / '__init__.py').exists():
    raise RuntimeError(
        'The JEPA side was not found. Clone it (https://github.com/alexmy21/ewm-jepa.git) '
        'near this project or set EWM_JEPA=/path/to/ewm-jepa.'
    )
sys.path.insert(0, str(EWM_JEPA))

EWM_SCENE_BIN = str(EWM_SM / 'target' / 'debug' / 'ewm-scene')
EWM_OPS_BIN = str(EWM_SM / 'target' / 'debug' / 'ewm-ops')
EWM_APP_BIN = str(EWM_SM / 'target' / 'debug' / 'ewm-app')
for binp, crate in [(EWM_SCENE_BIN, 'ewm-scene'), (EWM_OPS_BIN, 'ewm-ops'), (EWM_APP_BIN, 'ewm-app')]:
    if not os.path.exists(binp):
        subprocess.run(['cargo', 'build', '--quiet', '-p', crate], cwd=EWM_SM, check=True)

import torch
import torch.nn.functional as F
from ewm_jepa import JEPAPipeline, encode_masked, predict, target_encodings
import hllset_py

WORK = pathlib.Path('/tmp/ewm_jepa_coop')
WORK.mkdir(parents=True, exist_ok=True)


def ewm_scene(*args):
    r = subprocess.run([EWM_SCENE_BIN, *args], capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-800:])
    return json.loads(r.stdout.strip())


def ewm_ops(*args):
    r = subprocess.run([EWM_OPS_BIN, *args], capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-800:])
    return json.loads(r.stdout.strip())


def write_frames(path, token_lists):
    with open(path, 'w') as fh:
        for i, tokens in enumerate(token_lists):
            fh.write(json.dumps({'id': i + 1, 'tokens': tokens}) + '\n')
    return str(path)


print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('ewm-state-machine:', EWM_SM)
print('ewm-jepa:', EWM_JEPA)
print('work:', WORK)


torch 2.6.0+cu124 | cuda True
ewm-state-machine: /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine
ewm-jepa: /home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-jepa
work: /tmp/ewm_jepa_coop


## §1 — Pipeline validation (demo 01): IICA on real V-JEPA tids

The V-JEPA encoder turns a synthetic video into continuous patch encodings; the quantizer turns them into a `tid{n}` stream. The Rust lattice ingests that stream — and because ingestion is idempotent, immutable and content-addressed, ingesting the same stream twice returns the same G1/G2/G3 keys.

In [2]:
pipe = JEPAPipeline(codebook_size=2048, seed=0)
video, masks_enc, masks_pred = pipe.make_masks(batch_size=1)

with torch.no_grad():
    z_list = encode_masked(pipe.encoder, video, masks_enc)
zq = z_list[0][0]                       # (K, D) context encodings
ids = pipe.quantizer.encode(zq)         # (K,) tid indices
tokens = pipe.quantizer.ids_to_text(ids).split()
print('quantized stream:', len(tokens), 'tids,', int(torch.unique(ids).numel()), 'unique')

frames_path = write_frames(WORK / 'frames.jsonl', [tokens])
ing1 = ewm_scene('ingest', frames_path)
ing2 = ewm_scene('ingest', frames_path)

k1 = ing1['frames'][0]['key']
k2 = ing2['frames'][0]['key']
print('idempotent (same stream, same projection key):', k1 == k2)
print('content-addressed key:', k1)
print('projection popcount:', ing1['frames'][0]['pop'], 'bits')
print('G1/G2/G3: the three bootstrap LUTs live inside ingest (ng:G1, ng:G2, ng:G3)')

# The Python lattice (hllset_py) is a second, independent witness of IICA.
h = hllset_py.HLLSet.from_tokens(tokens)
print('hllset_py witness: popcount', h.popcount(), '| key', h.content_key())

INFO:root:MultiMaskWrapper(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed3D(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (blocks): ModuleList(
      (0-23): 24 x Block(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=Tr

INFO:root:PredictorMultiMaskWrapper(
  (backbone): VisionTransformerPredictor(
    (predictor_embed): Linear(in_features=1024, out_features=384, bias=True)
    (mask_tokens): ParameterList(
        (0): Parameter containing: [torch.float32 of size 1x1x384 (cuda:0)]
        (1): Parameter containing: [torch.float32 of size 1x1x384 (cuda:0)]
    )
    (predictor_blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_fea

INFO:root:Encoder number of parameters: 303885312


INFO:root:Predictor number of parameters: 22082944


quantized stream: 616 tids, 143 unique
idempotent (same stream, same projection key): True
content-addressed key: h:229103fd99b6e54267b827708f026eebcd226fec
projection popcount: 408 bits
G1/G2/G3: the three bootstrap LUTs live inside ingest (ng:G1, ng:G2, ng:G3)
hllset_py witness: popcount 141 | key h:201be7a91cf8beb64879ab62d047330dcd054499


/home/alexmy/.conda/envs/ewm-jepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


## §2 — Three-LUT unification (demo 02): the bootstrap views disambiguate

`ewm-scene ingest` registers every tid in three generation LUTs (G1/G2/G3). Ordered materialization walks the n-gram window chain across those LUTs — the cross-LUT disambiguation that restores the exact sequence.

In [3]:
mat = ewm_scene('materialize', frames_path)
restored = mat['frames'][0]['ordered']
loop_accuracy = 1.0 if restored == tokens else sum(a == b for a, b in zip(restored, tokens)) / max(len(tokens), 1)
set_restored = set(mat['frames'][0]['set'])
set_accuracy = len(set(tokens) & set_restored) / max(len(set(tokens)), 1)
print('ordered materialize:', restored[:8], '...' if len(restored) > 8 else '')
print('loop accuracy (ordered n-gram path):', round(loop_accuracy, 4))
print('set accuracy (the lattice stores every token):', set_accuracy)
print('cross-LUT disambiguation: the 1-gram collision set is resolved by the 2/3-gram LUTs')

ordered materialize: ['tid0', 'tid1000', 'tid1019', 'tid1020', 'tid1042', 'tid1074', 'tid1081', 'tid1083'] ...
loop accuracy (ordered n-gram path): 0.0016
set accuracy (the lattice stores every token): 1.0
cross-LUT disambiguation: the 1-gram collision set is resolved by the 2/3-gram LUTs


## §3 — Recursive IICA chain (demo 03): composition and fixed point

A [UM] node is the compound morphism `encodings → HLLSets → encodings`; chaining nodes composes IICA. The Rust [UM] loop runs the JEPA stream as turns, and a 3-node chain is composed inside one `ewm-ops` program via `call` — both converge to a fixed point.

In [4]:
# The [UM] loop: one turn = one quantized JEPA stream. The second identical
# turn brings no new bits and no basis change — the fixed point.
turn = ','.join(str(int(i)) for i in ids.tolist())
stub = f'{turn};{turn}'
repo = str(WORK / 'um_repo')
r = subprocess.run([EWM_APP_BIN, '--stub', stub, '--repo', repo],
                   capture_output=True, text=True, timeout=600)
print(r.stdout.strip()[-600:])

# The same chain as one ewm-ops program: base composed 3 times by call.
boot = f'''value s {' '.join(tokens)}
def base ( 1 -- 1 ) dup drop
def chain ( 1 -- 1 ) call:base call:base call:base
link value:@s -> in:chain.0
stack @s
fires 1
'''
boot_path = WORK / 'chain.ops'
boot_path.write_text(boot)
report = ewm_ops('--store', str(WORK / 'ops_chain'), '--boot', str(boot_path))
print('chain output == input (fixed point):', report['state'] == report['seed_stack'][0])

, "tid1671", "tid1671", "tid67", "tid2005", "tid1666", "tid4", "tid469", "tid989", "tid423", "tid255", "tid159", "tid1563", "tid67", "tid1563", "tid255", "tid1019", "tid67", "tid1541", "tid1389", "tid1863", "tid1863", "tid67", "tid1042", "tid1389", "tid1676", "tid67", "tid423", "tid1563", "tid1130", "tid1389", "tid1042"]
           commit=no-change (idempotent skip)
           tip=bf722fd1
           S(t) leaves=1 added=0 removed=0 retained=1
           full_image_tokens=143 bits(D=0 R=0 N=141)

final tip    : bf722fd1
context size : 141 bits (G1 lattice top)
Noether invariants (tip, G1): hold
chain output == input (fixed point): True


## §4 — [UM]-Net agent network (demo 04): fan-out by reference

Two agents (programs) consume the **same** JEPA stream. The operational graph passes CIDs, not copies: one value token routes the same reference to both agents, and the LIFO dispatcher fires them in deterministic order.

In [5]:
boot = f'''value s {' '.join(tokens)}
value ga ta
value gb tb
def agent_a ( 1 -- 1 ) @ga union
def agent_b ( 1 -- 1 ) @gb inter
link value:@s -> in:agent_a.0
link value:@s -> in:agent_b.0
stack @s
'''
boot_path = WORK / 'agents.ops'
boot_path.write_text(boot)
report = ewm_ops('--store', str(WORK / 'ops_agents'), '--boot', str(boot_path))
for rec in report['fire_log']['records']:
    print(f"fired {rec['op'][:12]}  in={len(rec['inputs'])}  out={len(rec['outputs'])}")
print('both agents consumed the same reference:',
      all(rec['inputs'] == [report['seed_stack'][0]] for rec in report['fire_log']['records']))

fired p:3b15880805  in=1  out=1
fired p:0e76086226  in=1  out=1
both agents consumed the same reference: True


## §5 — Holographic memory (demo 05): the temporal pyramid D/R/N

Each quantized JEPA step becomes a layer observation; the union stream splits into Departed / Retained / Novel against the previous state. The Noether balance `|N| − |D|` tracks how much the system replaces.

In [6]:
steps = []
for s in range(4):
    v, _, _ = pipe.make_masks(batch_size=1)
    with torch.no_grad():
        zz = encode_masked(pipe.encoder, v, masks_enc)[0][0]
    sid = pipe.quantizer.encode(zz)
    steps.append(pipe.quantizer.ids_to_text(sid).split())

pyramid_lines = []
for i, toks in enumerate(steps):
    pyramid_lines.append({'id': i + 1, 'perceptrons': {'perception': toks}})
pyramid_path = WORK / 'pyramid.jsonl'
with open(pyramid_path, 'w') as fh:
    for line in pyramid_lines:
        fh.write(json.dumps(line) + '\n')

pyr = ewm_scene('pyramid', str(pyramid_path))
print('step  R       D       N       |N|-|D|')
for i in range(len(pyr['drn']['dp'])):
    d, r_, n = pyr['drn']['dp'][i], pyr['drn']['rp'][i], pyr['drn']['np'][i]
    print(f'{i + 1:4d}  {r_:6d} {d:6d} {n:6d}  {abs(n - d):6d}')

step  R       D       N       |N|-|D|
   1     408      0      0       0
   2     408      0      0       0
   3     408      0      0       0


## §6 — Grounding proof (demo 06): fidelity round-trip and one-sided gating

**Proof 1 — fidelity.** The Rust lattice stores the **set** exactly; the Python De Bruijn path (the original demo 06) restores **order** for the predictor round-trip. Both numbers are reported.

**Proof 2 — grounding is one-sided.** Inject an unseen tid (`tid99999`) and run the gate as an `ewm-ops` program (`inter` with the codebook gate). The gated state is identical with or without the unseen tid: the gate keeps only what the codebook can express.

In [7]:
# Proof 1 — fidelity, each lattice doing what it does best.
# The Rust lattice stores the SET exactly (IICA); the Python De Bruijn
# path restores ORDER for the predictor round-trip.
mat = ewm_scene('materialize', frames_path)
set_restored = set(mat['frames'][0]['set'])
set_accuracy = len(set(tokens) & set_restored) / max(len(set(tokens)), 1)
print('Rust lattice set accuracy:', set_accuracy)

result = pipe.roundtrip(video, masks_enc, masks_pred)
s = result.stats[0]
print('JEPA round-trip (Python De Bruijn lattice):')
print('  retention    :', s.retention)
print('  quant cosine :', round(s.quant_cosine, 4))
print('  pred cosine  :', round(s.pred_cosine, 4))

# Proof 2 — the gate is one-sided: an unseen tid cannot pass it.
def gated_state(extra):
    stream = ' '.join(tokens + extra)
    gate = ' '.join(f'tid{i}' for i in range(2048))
    boot = f'''value s {stream}
value gate {gate}
def gate2 ( 2 -- 1 ) inter
link value:@s -> in:gate2.0
link value:@gate -> in:gate2.1
stack @s @gate
'''
    p = WORK / ('gate_clean.ops' if not extra else 'gate_injected.ops')
    p.write_text(boot)
    rep = ewm_ops('--store', str(WORK / ('gate_clean' if not extra else 'gate_injected')), '--boot', str(p))
    return rep['state']

clean = gated_state([])
injected = gated_state(['tid99999'])
print('gated(clean) == gated(+tid99999):', clean == injected)
print('  gated state:', clean)

Rust lattice set accuracy: 1.0


JEPA round-trip (Python De Bruijn lattice):
  retention    : 0.017857142857142856
  quant cosine : 0.0166
  pred cosine  : 0.3926
gated(clean) == gated(+tid99999): True
  gated state: h:201be7a91cf8beb64879ab62d047330dcd054499
